In [0]:
# Set our working environment
# Catalog and schema names
catalog_name = "formula1" #made up name for our company
schema_name = "formula1" #this our HR team's database

# Create the catalog if it does not exist
spark.sql(
    f"CREATE CATALOG IF NOT EXISTS {catalog_name}"
)

# Create the schema in the catalog
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}"
)

In [0]:
import os
import pandas as pd

# Local data folder
data_dir = "data/"

csv_files = {
    "circuits": "circuits.csv",
    "constructor_standings": "constructor_standings.csv", 
    "constructors": "constructors.csv",
    "driver_standings": "driver_standings.csv",
    "drivers": "drivers.csv",
    "lap_times": "lap_times.csv",
    "pit_stops": "pit_stops.csv",
    "qualifying": "qualifying.csv",
    "races": "races.csv",
    "results": "results.csv",
    "seasons": "seasons.csv",
    "sprint_results": "sprint_results.csv",
    "status": "status.csv",
}

# Pure Spark: Requires absolute paths, more complex file handling
# This hybrid: pandas handles file I/O easily, Spark handles Unity Catalog integration

# Load each CSV file
for table_name, file_name in csv_files.items():
    file_path = os.path.join(data_dir, file_name)
    
    # Read CSV and convert to Spark DataFrame in one line
    spark_df = spark.createDataFrame(pd.read_csv(file_path))
    spark_df.write.mode("overwrite").saveAsTable(f"{catalog_name}.formula1.{table_name}")

print("Tables created successfully")


In [0]:
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")
print(f"{catalog_name} catalog and {schema_name} schema have been created")

### **Apply Data Classifications and Tags**

First, we'll tag our tables with sensitivity levels. Table properties are metadata tags that help classify data sensitivity without changing the actual data structure:

- Metadata key-value pairs attached to tables
- Used by governance tools to understand data sensitivity
- Don't affect table structure or data
- Can be queried programmatically for compliance

Our data classification schema:
 - Public: Anyone can access 
 - Internal: Employees only 
 - Confidential: Limited access 
 - Restricted: Highly sensitive 

Adding classification tags like "Confidential," "Restricted," or "Public" to Unity Catalog serve as metadata that make your data assets more manageable, secure, and compliant with both internal policies and external regulations.


In [0]:
# Define the classification tags for each table
tables_config = [
    ("formula1.formula1.circuits", "Confidential", "true"),
    ("formula1.formula1.constructor_standings", "Confidential", "true"),
    ("formula1.formula1.constructors", "Confidential", "true"),
    ("formula1.formula1.driver_standings", "Confidential", "true"),
    ("formula1.formula1.drivers", "Confidential", "true"),
    ("formula1.formula1.lap_times", "Confidential", "true"),
    ("formula1.formula1.pit_stops", "Confidential", "true"),
    ("formula1.formula1.qualifying", "Confidential", "true"),
    ("formula1.formula1.races", "Confidential", "true"),
    ("formula1.formula1.results", "Confidential", "true"),
    ("formula1.formula1.seasons", "Confidential", "true"),
    ("formula1.formula1.sprint_results", "Confidential", "true"),
    ("formula1.formula1.status", "Confidential", "true")
]

# Apply classifications using tags (tags are metadata)
for table_name, classification, has_pii in tables_config:
    spark.sql(f"""
        ALTER TABLE {table_name} SET TAGS (
            'classification' = '{classification}',
            'contains_pii' = '{has_pii}'
        )
    """)
    print(f"✓ Classified {table_name} as {classification}")